In [1]:
import os


In [2]:
import cv2
import numpy as np

IMG_SIZE = 32

def detect_signs(frame):
    """Detects potential traffic sign regions using HSV color masking and contour geometry."""
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    
    red1 = cv2.inRange(hsv, (0, 130, 70), (10, 255, 255))
    red2 = cv2.inRange(hsv, (170, 130, 70), (180, 255, 255))
    mask = red1 | red2
    
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    boxes = []
    for c in contours:
        x, y, w, h = cv2.boundingRect(c)
        area = w * h
        aspect_ratio = float(w) / h
        
        if area > 4000 and 0.8 < aspect_ratio < 1.2:
            perimeter = cv2.arcLength(c, True)
            approx = cv2.approxPolyDP(c, 0.04 * perimeter, True)
            
            if 3 <= len(approx) <= 8:
                boxes.append((x, y, w, h))
                
    return boxes

def preprocess(frame, box):
    """Crops, resizes, and normalizes intensity of the detected sign region."""
    x, y, w, h = box
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    crop = gray[y:y + h, x:x + w]
    crop = cv2.resize(crop, (IMG_SIZE, IMG_SIZE))
    crop = cv2.equalizeHist(crop)
    return crop

print("✅ detect_signs and preprocess functions loaded successfully!")

✅ detect_signs and preprocess functions loaded successfully!


In [3]:
label = "Stop"         

folder = os.path.join("dataset", label)
os.makedirs(folder, exist_ok=True)
count = len(os.listdir(folder))

cv2.namedWindow("sign dataset", cv2.WINDOW_NORMAL)
cam = cv2.VideoCapture(0)
while True:
    ret, frame = cam.read()
    if not ret:
        break
    boxes = detect_signs(frame)
    for (x, y, w, h) in boxes:
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 0, 200), 2)
    cv2.putText(frame, f"{label}: {count}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.imshow("sign dataset", frame)

    key = cv2.waitKey(5) & 0xFF
    if key == ord('s') and boxes:
        cv2.imwrite(os.path.join(folder, f"{count}.png"), preprocess(frame, boxes[0]))
        count += 1
    if key == ord('q'):
        break

cam.release()
cv2.destroyAllWindows()

In [4]:
pip install tensorflow scikit-learn

Note: you may need to restart the kernel to use updated packages.


## Environment Setup & Dataset Analysis

In [5]:
import os
from collections import Counter

root_dir = "Indian_Traffic_Sign_Image_Dataset"

for root, dirs, files in os.walk(root_dir):
    print(root, "->", len(files), "files")

exts = Counter(os.path.splitext(f)[1].lower()
               for _, _, files in os.walk(root_dir) for f in files)
print(exts)

Indian_Traffic_Sign_Image_Dataset -> 3 files
Indian_Traffic_Sign_Image_Dataset\.cache -> 0 files
Indian_Traffic_Sign_Image_Dataset\.cache\huggingface -> 2 files
Indian_Traffic_Sign_Image_Dataset\.cache\huggingface\download -> 2 files
Indian_Traffic_Sign_Image_Dataset\.cache\huggingface\download\Annotations -> 0 files
Indian_Traffic_Sign_Image_Dataset\.cache\huggingface\download\Annotations\Annotations -> 150 files
Indian_Traffic_Sign_Image_Dataset\.cache\huggingface\download\images -> 0 files
Indian_Traffic_Sign_Image_Dataset\.cache\huggingface\download\images\images -> 150 files
Indian_Traffic_Sign_Image_Dataset\.cache\huggingface\trees -> 1 files
Indian_Traffic_Sign_Image_Dataset\Annotations -> 0 files
Indian_Traffic_Sign_Image_Dataset\Annotations\Annotations -> 150 files
Indian_Traffic_Sign_Image_Dataset\Images -> 0 files
Indian_Traffic_Sign_Image_Dataset\Images\0 -> 201 files
Indian_Traffic_Sign_Image_Dataset\Images\1 -> 201 files
Indian_Traffic_Sign_Image_Dataset\Images\10 -> 201 

In [6]:
import os
from collections import Counter

root_dir = "Indian_Traffic_Sign_Image_Dataset"
print("exists:", os.path.exists(root_dir))


n = 0
for root, dirs, files in os.walk(root_dir):
    print(root, "->", len(files), "files")
    n += 1
    if n >= 25:
        break


all_files = [f for _, _, fs in os.walk(root_dir) for f in fs]
print(Counter(os.path.splitext(f)[1].lower() for f in all_files))
print(all_files[:10])

exists: True
Indian_Traffic_Sign_Image_Dataset -> 3 files
Indian_Traffic_Sign_Image_Dataset\.cache -> 0 files
Indian_Traffic_Sign_Image_Dataset\.cache\huggingface -> 2 files
Indian_Traffic_Sign_Image_Dataset\.cache\huggingface\download -> 2 files
Indian_Traffic_Sign_Image_Dataset\.cache\huggingface\download\Annotations -> 0 files
Indian_Traffic_Sign_Image_Dataset\.cache\huggingface\download\Annotations\Annotations -> 150 files
Indian_Traffic_Sign_Image_Dataset\.cache\huggingface\download\images -> 0 files
Indian_Traffic_Sign_Image_Dataset\.cache\huggingface\download\images\images -> 150 files
Indian_Traffic_Sign_Image_Dataset\.cache\huggingface\trees -> 1 files
Indian_Traffic_Sign_Image_Dataset\Annotations -> 0 files
Indian_Traffic_Sign_Image_Dataset\Annotations\Annotations -> 150 files
Indian_Traffic_Sign_Image_Dataset\Images -> 0 files
Indian_Traffic_Sign_Image_Dataset\Images\0 -> 201 files
Indian_Traffic_Sign_Image_Dataset\Images\1 -> 201 files
Indian_Traffic_Sign_Image_Dataset\Imag

## Preprocessing & Region-of-Interest Extraction

In [7]:
import os, cv2
import xml.etree.ElementTree as ET
from collections import Counter

BASE = "Indian_Traffic_Sign_Image_Dataset"
IMG_DIR = os.path.join(BASE, "images", "images")
XML_DIR = os.path.join(BASE, "Annotations", "Annotations")

counts = Counter()
missing = 0

for xml_name in os.listdir(XML_DIR):
    if not xml_name.endswith(".xml"):
        continue
    stem = os.path.splitext(xml_name)[0]
    img_path = os.path.join(IMG_DIR, stem + ".jpg")
    if not os.path.exists(img_path):
        missing += 1
        continue

    img = cv2.imread(img_path)
    if img is None:
        continue
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    H, W = gray.shape

    root = ET.parse(os.path.join(XML_DIR, xml_name)).getroot()
    for obj in root.findall("object"):
        name = obj.find("name").text.strip().replace(" ", "_").replace("/", "_")
        bb = obj.find("bndbox")
        x1 = max(0, int(float(bb.find("xmin").text)))
        y1 = max(0, int(float(bb.find("ymin").text)))
        x2 = min(W, int(float(bb.find("xmax").text)))
        y2 = min(H, int(float(bb.find("ymax").text)))
        if x2 - x1 < 10 or y2 - y1 < 10:
            continue                                        # skip tiny / bad boxes

        crop = gray[y1:y2, x1:x2]                           # sign cut
        crop = cv2.resize(crop, (IMG_SIZE, IMG_SIZE))       # resize
        crop = cv2.equalizeHist(crop)                       # normalize intensity

        out_dir = os.path.join("dataset", name)
        os.makedirs(out_dir, exist_ok=True)
        cv2.imwrite(os.path.join(out_dir, f"hf_{stem.replace(' ', '_')}_{counts[name]}.png"), crop)
        counts[name] += 1

print("images without a matching xml/jpg:", missing)
for k, v in counts.most_common():
    print(f"{k:30s} {v}")

images without a matching xml/jpg: 150


In [8]:
import os
import cv2

dataset_dir = "dataset"

if not os.path.exists(dataset_dir):
    print("❌ 'dataset/' folder does not exist!")
else:
    print("📁 Checking folders inside 'dataset/':\n")
    for name in sorted(os.listdir(dataset_dir)):
        p = os.path.join(dataset_dir, name)
        if os.path.isdir(p) and not name.startswith("."):
            files = [f for f in os.listdir(p) if not f.startswith(".")]
           
            valid_count = 0
            for f in files:
                img_path = os.path.join(p, f)
                if cv2.imread(img_path, cv2.IMREAD_GRAYSCALE) is not None:
                    valid_count += 1
            print(f"  - Class '{name}': {len(files)} total files, {valid_count} readable images")

📁 Checking folders inside 'dataset/':

  - Class 'Stop': 1 total files, 1 readable images
  - Class 'traffic_sign': 194 total files, 194 readable images


## Convolutional Neural Network (CNN) Model Training

In [9]:
import json
import os
import cv2
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

IMG_SIZE = 32  
MIN_IMAGES = 1  
classes = []
for c in sorted(os.listdir("dataset")):
    p = os.path.join("dataset", c)
    if os.path.isdir(p) and not c.startswith("."):
        n = len(os.listdir(p))
        if n >= MIN_IMAGES:
            classes.append(c)
        else:
            print(f"skipping {c} ({n} images)")
print("Classes:", classes)

assert len(classes) > 0, "dataset/ is empty - run the XML crop cell first"


X, y = [], []
for idx, name in enumerate(classes):
    class_folder = os.path.join("dataset", name)
    for f in os.listdir(class_folder):
        if f.startswith("."):
            continue
        img_path = os.path.join(class_folder, f)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            # Force resize to guarantee uniform dimensions
            if img.shape != (IMG_SIZE, IMG_SIZE):
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            X.append(img)
            y.append(idx)

X = np.array(X, dtype="float32").reshape(-1, IMG_SIZE, IMG_SIZE, 1) / 255.0
y = np.array(y, dtype="int32")
print("Total images loaded:", len(X))
assert len(X) > 0, "dataset/ is empty - run the XML crop cell first"


if len(classes) > 1 and len(X) >= 10:
    try:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=42)
    except ValueError:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42)
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42)

aug = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
])


num_classes = len(classes)
output_units = num_classes if num_classes > 1 else 1
output_activation = "softmax" if num_classes > 1 else "sigmoid"
loss_fn = "sparse_categorical_crossentropy" if num_classes > 1 else "binary_crossentropy"

model = tf.keras.Sequential([
    tf.keras.Input((IMG_SIZE, IMG_SIZE, 1)), aug,
    tf.keras.layers.Conv2D(32, 3, activation="relu"), tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation="relu"), tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(), tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(output_units, activation=output_activation),
])

model.compile(optimizer="adam", loss=loss_fn, metrics=["accuracy"])

stop = tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=8,
                                        restore_best_weights=True)
model.fit(X_train, y_train, epochs=60, batch_size=32,
          validation_data=(X_test, y_test), callbacks=[stop])

loss, acc = model.evaluate(X_test, y_test)
print(f"Test accuracy: {acc:.2%}")

model.save("sign_model.keras")
json.dump(classes, open("classes.json", "w"))
print("✅ Model trained and saved successfully!")

Classes: ['Stop', 'traffic_sign']
Total images loaded: 195
Epoch 1/60
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 131ms/step - accuracy: 0.8013 - loss: 0.3200 - val_accuracy: 1.0000 - val_loss: 0.0014
Epoch 2/60
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.9936 - loss: 0.0723 - val_accuracy: 1.0000 - val_loss: 3.3011e-06
Epoch 3/60
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.9936 - loss: 0.1128 - val_accuracy: 1.0000 - val_loss: 8.0390e-07
Epoch 4/60
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 0.9936 - loss: 0.1006 - val_accuracy: 1.0000 - val_loss: 3.8550e-05
Epoch 5/60
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 0.9936 - loss: 0.0630 - val_accuracy: 1.0000 - val_loss: 8.3359e-04
Epoch 6/60
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.9936 - loss: 0.0431 - val_accuracy: 1.0000 - val_loss: 0.0046
Epoch 7/60
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.9936 - loss: 0.0469 - val_accuracy: 1.0000 - val_loss: 0.0105
Epoch 8/60
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step 

In [10]:
import os
print("Working folder:", os.getcwd())
print("Dataset exists:", os.path.exists("dataset"))

for name in os.listdir("dataset"):
    path = os.path.join("dataset", name)
    if os.path.isdir(path):
        print(name, "->", len(os.listdir(path)), "files")

Working folder: c:\Users\Vishal Verma\OneDrive\Desktop\AI_TSR
Dataset exists: True
Stop -> 1 files
traffic_sign -> 194 files


In [11]:
import os

print("Searching for folders containing .jpg files:")
for root, dirs, files in os.walk("Indian_Traffic_Sign_Image_Dataset"):
    jpgs = [f for f in files if f.endswith(".jpg")]
    if len(jpgs) > 0:
        print(f"👉 Found {len(jpgs)} .jpg files in: {root}")

Searching for folders containing .jpg files:
👉 Found 1 .jpg files in: Indian_Traffic_Sign_Image_Dataset\speed_limit_50


In [12]:
import os
import random
import cv2
import json
import tensorflow as tf

model = tf.keras.models.load_model("sign_model.keras")
classes = json.load(open("classes.json"))


IMG_DIR = os.path.join("Indian_Traffic_Sign_Image_Dataset", "images", "images")

if not os.path.exists(IMG_DIR):
    
    for root, dirs, files_list in os.walk("Indian_Traffic_Sign_Image_Dataset"):
        if any(f.endswith(".jpg") for f in files_list):
            IMG_DIR = root
            break

print(f"📁 Using image directory: {IMG_DIR}")

files = [f for f in os.listdir(IMG_DIR) if f.endswith(".jpg")]
assert len(files) > 0, f"❌ No .jpg files found in {IMG_DIR}! Please check the folder path."

for f in random.sample(files, min(3, len(files))):
    frame = cv2.imread(os.path.join(IMG_DIR, f))
    if frame is None:
        continue
    for box in detect_signs(frame):
        x, y, w, h = box
        img = preprocess(frame, box).astype("float32").reshape(1, IMG_SIZE, IMG_SIZE, 1) / 255.0
        probs = model(img, training=False).numpy()[0]
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 0, 200), 3)
        cv2.putText(frame, f"{classes[probs.argmax()]} {probs.max():.0%}", (x, max(y - 10, 20)),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 200), 3)
    cv2.imshow("test", cv2.resize(frame, None, fx=0.4, fy=0.4))
    cv2.waitKey(0)

cv2.destroyAllWindows()
cv2.waitKey(1)

📁 Using image directory: Indian_Traffic_Sign_Image_Dataset\speed_limit_50


-1

In [13]:
model = tf.keras.models.load_model("sign_model.keras")
classes = json.load(open("classes.json"))

cam = cv2.VideoCapture(0, cv2.CAP_DSHOW)
cam.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cam.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cv2.namedWindow("traffic sign recognition", cv2.WINDOW_NORMAL)

try:
    while True:
        ret, frame = cam.read()
        if not ret:
            break
        for box in detect_signs(frame):
            x, y, w, h = box
            img = preprocess(frame, box).astype("float32").reshape(1, IMG_SIZE, IMG_SIZE, 1) / 255.0
            probs = model(img, training=False).numpy()[0]
            if probs.max() > 0.8:
                cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 0, 200), 2)
                cv2.putText(frame, f"{classes[probs.argmax()]} {probs.max():.0%}", (x, max(y - 8, 20)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 200), 2)
        cv2.imshow("traffic sign recognition", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
finally:
    cam.release()
    cv2.destroyAllWindows()
    cv2.waitKey(1)